In [1]:
import os
import pickle
import numpy as np
import faiss
from pathlib import Path
import torch
from transformers import AutoProcessor, CLIPModel

In [2]:
PKL_FILES_DIRECTORY = "../"
OUTPUT_DIRECTORY = "faiss_index"
INDEX_TYPE = "flat"  # Options: 'flat', 'ivf', 'hnsw'

### load embeddings from pkl files

In [4]:
def load_video_embeddings(pkl_files_directory):
    embeddings = []
    metadata = []
    
    pkl_files = list(Path(pkl_files_directory).glob("*.pkl"))
    print(f"Found {len(pkl_files)} pickle files")
    
    for pkl_file in pkl_files:
        try:
            with open(pkl_file, 'rb') as f:
                data = pickle.load(f)
                
            # Extract embedding (assuming it's the average embedding)
            for item in data:
                embedding = item.get('embedding')
                if embedding is not None:
                # Convert to numpy array if it's a list
                    if isinstance(embedding, list):
                        embedding = np.array(embedding, dtype=np.float32)
                    
                    embeddings.append(embedding)
                    # Store metadata
                    metadata.append({'video_id': item.get('video_id', 'unknown'), 'video_path': item.get('video_path', ''), 'thumbnail_path': item.get('thumbnail_path', ''),})
        except Exception as e:
            print(f"Error loading {pkl_file}: {e}")
            continue
    
    print(f"Successfully loaded {len(embeddings)} video embeddings")
    return embeddings, metadata

In [5]:
%%time
embeddings, metadata = load_video_embeddings(PKL_FILES_DIRECTORY)

Found 1 pickle files
Successfully loaded 1472 video embeddings
CPU times: user 96.7 ms, sys: 20.2 ms, total: 117 ms
Wall time: 115 ms


### create in index

In [6]:
def create_faiss_index(embeddings, index_type='flat'):
    if not embeddings:
        raise ValueError("No embeddings provided")
    
    # Convert to numpy array
    embeddings_matrix = np.vstack(embeddings).astype(np.float32)
    dimension = embeddings_matrix.shape[1]
    
    print(f"Creating FAISS index with {embeddings_matrix.shape[0]} vectors of dimension {dimension}")
    
    index = faiss.IndexFlatIP(dimension)  # Inner Product (cosine similarity for normalized vectors)
    # Normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings_matrix)
    index.add(embeddings_matrix)
    
    print(f"Index created successfully with {index.ntotal} vectors")
    return index

In [7]:
%%time
index = create_faiss_index(embeddings, INDEX_TYPE)

Creating FAISS index with 1472 vectors of dimension 512
Index created successfully with 1472 vectors
CPU times: user 15.6 ms, sys: 11.1 ms, total: 26.8 ms
Wall time: 25.3 ms


### store in index

In [8]:
def save_index_and_metadata(index, metadata, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    # Save FAISS index
    index_path = os.path.join(output_dir, 'video_index.faiss')
    faiss.write_index(index, index_path)
    print(f"FAISS index saved to: {index_path}")
    
    # Save metadata
    metadata_path = os.path.join(output_dir, 'video_metadata.pkl')
    with open(metadata_path, 'wb') as f:
        pickle.dump(metadata, f)

    print(f"Metadata saved to: {metadata_path}")
    
    # Save index info
    info_path = os.path.join(output_dir, 'index_info.txt')
    with open(info_path, 'w') as f:
        f.write(f"Total videos indexed: {len(metadata)}\n")
        f.write(f"Index type: {type(index).__name__}\n")
        f.write(f"Vector dimension: {index.d}\n")
        f.write(f"Total vectors: {index.ntotal}\n")
        
    print(f"Index info saved to: {info_path}")

In [9]:
%%time
save_index_and_metadata(index, metadata, OUTPUT_DIRECTORY)

FAISS index saved to: faiss_index/video_index.faiss
Metadata saved to: faiss_index/video_metadata.pkl
Index info saved to: faiss_index/index_info.txt
CPU times: user 2.91 ms, sys: 4.02 ms, total: 6.93 ms
Wall time: 6.43 ms
